In [65]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

PROCESSING=False 
PROCESSING=True #set to True if using Turbulence-Removed Binary Arrays
Processed_string = GetProcessedString(PROCESSING=PROCESSING)

In [66]:
####################################
#ENVIRONMENT SETUP

In [67]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import warnings

In [68]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [69]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [70]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [71]:
####################################
#LOADING CLASSES

In [73]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation_DivideMassFlux",
                                dtype='int32')

=== CM1 Data Summary ===
 Simulation #:   1
 Resolution:     1km
 Time step:      5min
 Vertical levels:34
 Parcels:        1e6
 Data file:      /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Test_Simulation/cm1out_1km_5min_34nz.nc
 Parcel file:    /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Test_Simulation/cm1out_pdata_1km_5min_1e6np.nc
 Time steps:     133

=== DataManager Summary ===
 inputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData
 outputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/EntrainmentCalculation
 inputDataDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData/1km_5min_34nz/ModelData
 inputParcelDirectory #: 

In [74]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

Running timesteps from 0:6 



In [75]:
####################################
#FUNCTIONS

In [76]:
#CALCULATING ENTRAINMENT
def GetVariable_T(t, varName, cache=None): #*MassFlux
    """Load variable for a given timestep, using cached open HDF5 files if available."""
    if cache is None:
        cache = {}

    timeString = ModelData.timeStrings[t]

    # If file for this timestep already in cache, reuse it
    if timeString in cache:
        f = cache[timeString]
    else:
        # Otherwise, open new file and cache it
        f = DataManager.GetTimestepData_V2(DataManager.inputDataDirectory, timeString)
        cache[timeString] = f

    # Load the desired variable from the open file (lazy read)
    if varName in f.keys():
        output = f[varName][:]  # read actual data
    else:
        # fallback for derived variables
        output = CallVariable(ModelData, DataManager, timeString, variableName=varName)
    return output

def SafeDivide(numerator, denominator):
    # 1. Create an array of NaNs with the same shape as the numerator
    result = np.full(numerator.shape, np.nan, dtype=np.float32)
    
    # 2. Find where we can safely divide
    nonzero = (denominator != 0)
    
    # 3. Only fill the "nonzero" indices with the division result
    result[nonzero] = numerator[nonzero] / denominator[nonzero]
    
    return result
    
# def LoadMassFlux(t): #*MassFlux
#     """
#     lagrangian version
#     """
    
#     updraftType = "g" 
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux_g = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     updraftType = "c" 
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux_c = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     return MassFlux_g, MassFlux_c

def LoadMassFlux(t):
    """
    eulerian version
    """
    
    timeString=ModelData.timeStrings[t]

    varName='rho'
    rho = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='winterp'
    w = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='A_g'
    A_g = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    varName='A_c'
    A_c = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    
    MassFlux_g = rho*(A_g*1)*w
    MassFlux_c = rho*(A_c*1)*w
    return MassFlux_g,MassFlux_c

def DivideByMassFlux(t, varName,
                     MassFlux_g, MassFlux_c):
    variable = GetVariable_T(t, varName)
    MassFlux = MassFlux_g if "g" in varName else MassFlux_c
    variable_DivideMassFlux = SafeDivide(variable, MassFlux)
    return variable_DivideMassFlux

In [77]:
def GetVarNames_Entrainment(Processed_string):
    varNames = [f"{Processed_string}Entrainment_g",
                f"{Processed_string}Entrainment_c"]
    
    varNames += [f"{Processed_string}TransferEntrainment_g",
                f"{Processed_string}TransferEntrainment_c"]
    return varNames

def GetVarNames_Detrainment(Processed_string):
    varNames = [f"{Processed_string}Detrainment_g",
                f"{Processed_string}Detrainment_c"]
    
    varNames += [f"{Processed_string}TransferDetrainment_g",
                f"{Processed_string}TransferDetrainment_c"]
    return varNames

def AddDivideMassFluxString(varName):
    parts = varName.rsplit('_', 1)
    varName_DivideMassFlux = f"{parts[0]}_DivideMassFlux_{parts[1]}"
    return varName_DivideMassFlux

In [78]:
def RunCalculation_Entrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Entrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

def RunCalculation_Detrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Detrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

In [79]:
##############################################
#RUNNING

In [ ]:
#running calculation
for t in tqdm(loop_elements, total=len(loop_elements)):
    # if np.mod(t,1)==0: print(f'Current time {t}')
    if t == ModelData.Ntime-1:
        continue

    #loading MassFlux
    [MassFlux_g, MassFlux_c] =  LoadMassFlux(t)

    #calculating variables
    outputDictionary_Entrainment_DivideMassFlux = RunCalculation_Entrainment(t, Processed_string,
                                                                         MassFlux_g, MassFlux_c)
    outputDictionary_Detrainment_DivideMassFlux = RunCalculation_Detrainment(t, Processed_string,
                                                                             MassFlux_g, MassFlux_c)
    
    #outputting
    timeString = ModelData.timeStrings[t]
    
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Entrainment_DivideMassFlux, dataName=f"{Processed_string}Entrainment_DivideMassFlux",dtype='float32')

    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Detrainment_DivideMassFlux, dataName=f"{Processed_string}Detrainment_DivideMassFlux",dtype='float32')

In [86]:
########################
#TESTING

In [81]:
#FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#FUNCTIONS

#APPLYING ENTRAINMENT CONSTANT
entrainmentConstant = DataManager.LoadCalculations(
    DataManager.outputDataDirectory.replace('EntrainmentCalculation_DivideMassFlux','EntrainmentCalculation'),
    dataName="EntrainmentConstant",
    verbose=False,
)["entrainmentConstant"]
def ApplyEntrainmentConstant(variable):
    """
    Multiply each array in the input dictionary by the 1D entrainment constant profile.
    Returns the processed arrays in the same order as the input dictionary.
    """
    
    # Return arrays in the same order as input
    return variable*entrainmentConstant[:, np.newaxis, np.newaxis]

#APPLYING MASS FLUX CONSTANT
massFluxConstant = DataManager.LoadCalculations(
    os.path.join(os.path.dirname(DataManager.outputDataDirectory), "MassFluxCalculation"),
    dataName="MassFluxConstant",
    verbose=False,
)["massFluxConstant"]
def ApplyMassFluxConstant(variable): #*MassFlux
    """
    Multiply each array in the input dictionary by the 1D mass flux constant profile.
    Returns the processed arrays in the same order as the input dictionary.
    """

    # Return arrays in the same order as input
    return variable * massFluxConstant[:, np.newaxis, np.newaxis]

def LoadMassFlux_lagrangian(t,updraftType="c"): #*MassFlux
    """
    lagrangian version
    """
    
    massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
    MassFlux = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

    return MassFlux

def CalculateMassFlux_eulerian(t,updraftType="c"):
    
    timeString=ModelData.timeStrings[t]

    varName='rho'
    rho = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='winterp'
    w = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    varName=f'A_{updraftType}'
    A = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)

    MassFlux = rho*(A*1)*w
    return MassFlux

def GetData_multipletimes(ts, updraftType="c"):

    varName = f"{Processed_string}Entrainment_{updraftType}"
    
    # Get shape from a sample
    sample = GetVariable_T(ts[0], varName)
    Nz = sample.shape[0]

    # Pre-allocate (time, z)
    variable_array = np.full((len(ts), Nz), np.nan)
    MassFlux_lagrangian_array = np.full((len(ts), Nz), np.nan)
    MassFlux_eulerian_array = np.full((len(ts), Nz), np.nan)

    for i, t in tqdm(enumerate(ts), total=len(ts)):
        v = GetVariable_T(t, varName)
        v = ApplyEntrainmentConstant(v)
        
        mfl = LoadMassFlux_lagrangian(t,updraftType)
        mfl = ApplyMassFluxConstant(mfl)
        mfe = CalculateMassFlux_eulerian(t,updraftType)

        # --- Average over x,y (axis 1,2) ---
        variable_array[i] = np.nanmean(v, axis=(1, 2))
        MassFlux_lagrangian_array[i] = np.nanmean(mfl, axis=(1, 2))
        MassFlux_eulerian_array[i] = np.nanmean(mfe, axis=(1, 2))

    return variable_array, MassFlux_lagrangian_array, MassFlux_eulerian_array

In [82]:
#DATA LOADING

def SafeAdd(sumArray, countArray, newArray):
    valid = np.isfinite(newArray)
    sumArray[valid] += newArray[valid]
    countArray[valid] += 1
    return sumArray, countArray


def CalculateMeans_MultipleTimes(timeTuple,updraftType="c"):
    [t1, t2] = timeTuple
    timeIndices = np.arange(t1, t2+1)
    Nt = len(timeIndices)
    
    # --- INIT STORAGE (t,z) ---
    Entrainment_mean = None
    MF_lagrangian_mean = None
    MF_eulerian_mean = None
    
    Entrainment_over_MF_lagrangian_mean = None
    Entrainment_over_MF_eulerian_mean = None
    
    Entrainment_over_MF_lagrangian_ratio_of_means = None
    Entrainment_over_MF_eulerian_ratio_of_means = None

    # --- LOOP ---
    for i, t in enumerate(tqdm(timeIndices)):
        
        # --- LOAD ---
        varName = f"{Processed_string}Entrainment_{updraftType}"
        Entrainment = GetVariable_T(t, varName)
        Entrainment = ApplyEntrainmentConstant(Entrainment)
        
        MF_lagrangian = LoadMassFlux_lagrangian(t, updraftType)
        MF_lagrangian = ApplyMassFluxConstant(MF_lagrangian)
        
        MF_eulerian = CalculateMassFlux_eulerian(t, updraftType)
        
        # --- MEANS (per time) ---
        Entrainment_mean_t = np.nanmean(Entrainment, axis=(1,2))
        MF_lagrangian_mean_t = np.nanmean(MF_lagrangian, axis=(1,2))
        MF_eulerian_mean_t   = np.nanmean(MF_eulerian, axis=(1,2))
        
        # --- MEAN OF RATIOS (per time) ---
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            ratio_L_t = np.nanmean(SafeDivide(Entrainment, MF_lagrangian), axis=(1,2))
            ratio_E_t = np.nanmean(SafeDivide(Entrainment, MF_eulerian), axis=(1,2))
        
        # --- RATIO OF MEANS (FIXED: consistent mask) ---
        valid_L = np.isfinite(Entrainment) & np.isfinite(MF_lagrangian)
        valid_E = np.isfinite(Entrainment) & np.isfinite(MF_eulerian)

        E_sum_L = np.nansum(np.where(valid_L, Entrainment, 0), axis=(1,2))
        M_sum_L = np.nansum(np.where(valid_L, MF_lagrangian, 0), axis=(1,2))

        E_sum_E = np.nansum(np.where(valid_E, Entrainment, 0), axis=(1,2))
        M_sum_E = np.nansum(np.where(valid_E, MF_eulerian, 0), axis=(1,2))

        ratio_of_means_L_t = SafeDivide(E_sum_L, M_sum_L)
        ratio_of_means_E_t = SafeDivide(E_sum_E, M_sum_E)
        
        # --- INIT ARRAYS ---
        if Entrainment_mean is None:
            Nz = len(Entrainment_mean_t)
            
            Entrainment_mean = np.full((Nt, Nz), np.nan)
            MF_lagrangian_mean = np.full((Nt, Nz), np.nan)
            MF_eulerian_mean   = np.full((Nt, Nz), np.nan)
            
            Entrainment_over_MF_lagrangian_mean = np.full((Nt, Nz), np.nan)
            Entrainment_over_MF_eulerian_mean   = np.full((Nt, Nz), np.nan)
            
            Entrainment_over_MF_lagrangian_ratio_of_means = np.full((Nt, Nz), np.nan)
            Entrainment_over_MF_eulerian_ratio_of_means   = np.full((Nt, Nz), np.nan)
        
        # --- STORE ---
        Entrainment_mean[i, :] = Entrainment_mean_t
        MF_lagrangian_mean[i, :] = MF_lagrangian_mean_t
        MF_eulerian_mean[i, :]   = MF_eulerian_mean_t
        
        Entrainment_over_MF_lagrangian_mean[i, :] = ratio_L_t
        Entrainment_over_MF_eulerian_mean[i, :]   = ratio_E_t
        
        Entrainment_over_MF_lagrangian_ratio_of_means[i, :] = ratio_of_means_L_t
        Entrainment_over_MF_eulerian_ratio_of_means[i, :]   = ratio_of_means_E_t

    return (
        Entrainment_mean,
        MF_lagrangian_mean,
        MF_eulerian_mean,
        Entrainment_over_MF_lagrangian_mean,
        Entrainment_over_MF_eulerian_mean,
        Entrainment_over_MF_lagrangian_ratio_of_means,
        Entrainment_over_MF_eulerian_ratio_of_means
    )

def PlotMeans(
    Entrainment_mean,
    MF_lagrangian_mean,
    MF_eulerian_mean,
    Entrainment_over_MF_lagrangian_mean,
    Entrainment_over_MF_eulerian_mean,
    Entrainment_over_MF_lagrangian_ratio_of_means,
    Entrainment_over_MF_eulerian_ratio_of_means,
    plotType="line"
):
    
    # --- Figure Setup ---
    fig, axes = plt.subplots(
        2, 4,
        figsize=(18, 8),
        sharey=True,
        sharex='col',
        gridspec_kw={'wspace': 0.05, 'hspace': 0.1}
    )
    
    color = 'k'
    
    # =========================
    # LINE PLOTS (mean over time)
    # =========================
    if plotType == "line":
        
        # take mean over time (axis=0)
        Entrainment_mean = np.nanmean(Entrainment_mean, axis=0)
        MF_lagrangian_mean = np.nanmean(MF_lagrangian_mean, axis=0)
        MF_eulerian_mean   = np.nanmean(MF_eulerian_mean, axis=0)
        
        Entrainment_over_MF_lagrangian_mean = np.nanmean(Entrainment_over_MF_lagrangian_mean, axis=0)
        Entrainment_over_MF_eulerian_mean   = np.nanmean(Entrainment_over_MF_eulerian_mean, axis=0)
        
        Entrainment_over_MF_lagrangian_ratio_of_means = np.nanmean(
            Entrainment_over_MF_lagrangian_ratio_of_means, axis=0
        )
        Entrainment_over_MF_eulerian_ratio_of_means = np.nanmean(
            Entrainment_over_MF_eulerian_ratio_of_means, axis=0
        )
        
        # --- Entrainment ---
        for axis in [axes[0,0], axes[1,0]]:
            axis.plot(Entrainment_mean, ModelData.zh, color=color)
            axis.set_title("<E>")
        
        # --- Mass Flux ---
        axes[0,1].plot(MF_lagrangian_mean, ModelData.zh, color=color)
        axes[0,1].set_title("<M> (lagrangian)")
        
        axes[1,1].plot(MF_eulerian_mean, ModelData.zh, color=color)
        axes[1,1].set_title("<M> (eulerian)")
        
        # --- <E/M> ---
        axes[0,2].plot(Entrainment_over_MF_lagrangian_mean, ModelData.zh, color=color)
        axes[0,2].set_title("<E/M>")
        
        axes[1,2].plot(Entrainment_over_MF_eulerian_mean, ModelData.zh, color=color)
        axes[1,2].set_title("<E/M>")
        
        # --- <E>/<M> ---
        axes[0,3].plot(Entrainment_over_MF_lagrangian_ratio_of_means, ModelData.zh, color=color)
        axes[0,3].set_title("<E>/<M>")
        
        axes[1,3].plot(Entrainment_over_MF_eulerian_ratio_of_means, ModelData.zh, color=color)
        axes[1,3].set_title("<E>/<M>")
    
    # =========================
    # CONTOUR PLOTS (time-height)
    # =========================
    elif plotType == "contour":
        
        percentile_min = 1
        percentile_max = 99
        nLevels = 20
    
        # --- DEFINE LEVELS PER COLUMN ---
        # Column 1: Entrainment
        levels1 = np.linspace(
            np.nanpercentile(Entrainment_mean, percentile_min),
            np.nanpercentile(Entrainment_mean, percentile_max),
            nLevels
        )
    
        # Column 2: Mass Flux (L + E together)
        levels2 = np.linspace(
            np.nanpercentile(
                np.concatenate([MF_lagrangian_mean.ravel(), MF_eulerian_mean.ravel()]),
                percentile_min
            ),
            np.nanpercentile(
                np.concatenate([MF_lagrangian_mean.ravel(), MF_eulerian_mean.ravel()]),
                percentile_max
            ),
            nLevels
        )
    
        # Column 3: <E/M>
        levels3 = np.linspace(
            np.nanpercentile(
                np.concatenate([
                    Entrainment_over_MF_lagrangian_mean.ravel(),
                    Entrainment_over_MF_eulerian_mean.ravel()
                ]),
                percentile_min
            ),
            np.nanpercentile(
                np.concatenate([
                    Entrainment_over_MF_lagrangian_mean.ravel(),
                    Entrainment_over_MF_eulerian_mean.ravel()
                ]),
                percentile_max
            ),
            nLevels
        )
    
        # Column 4: <E>/<M>
        levels4 = np.linspace(
            np.nanpercentile(
                np.concatenate([
                    Entrainment_over_MF_lagrangian_ratio_of_means.ravel(),
                    Entrainment_over_MF_eulerian_ratio_of_means.ravel()
                ]),
                percentile_min
            ),
            np.nanpercentile(
                np.concatenate([
                    Entrainment_over_MF_lagrangian_ratio_of_means.ravel(),
                    Entrainment_over_MF_eulerian_ratio_of_means.ravel()
                ]),
                percentile_max
            ),
            nLevels
        )
    
        # --- HELPER ---
        def PlotContour(axis, data, levels, title):
            T = np.arange(data.shape[0])
            cf = axis.contourf(T, ModelData.zh, data.T, levels=levels, cmap='turbo', extend='both')
            axis.set_title(title)
            return cf
    
        # --- COLUMN 1 ---
        cf = PlotContour(axes[0,0], Entrainment_mean, levels1, "<E>")
        cbar1=fig.colorbar(cf, ax=axes[0,0])
        
        cf = PlotContour(axes[1,0], Entrainment_mean, levels1, "<E>")
        cbar2=fig.colorbar(cf, ax=axes[1,0])
    
        # --- COLUMN 2 ---
        cf = PlotContour(axes[0,1], MF_lagrangian_mean, levels2, "<M> (lagrangian)")
        cbar3=fig.colorbar(cf, ax=axes[0,1])
        
        cf = PlotContour(axes[1,1], MF_eulerian_mean, levels2, "<M> (eulerian)")
        cbar4=fig.colorbar(cf, ax=axes[1,1])
    
        # --- COLUMN 3 ---
        cf = PlotContour(axes[0,2], Entrainment_over_MF_lagrangian_mean, levels3, "<E/M>")
        cbar5=fig.colorbar(cf, ax=axes[0,2])
        
        cf = PlotContour(axes[1,2], Entrainment_over_MF_eulerian_mean, levels3, "<E/M>")
        cbar6=fig.colorbar(cf, ax=axes[1,2])
    
        # --- COLUMN 4 ---
        cf = PlotContour(axes[0,3], Entrainment_over_MF_lagrangian_ratio_of_means, levels4, "<E>/<M>")
        cbar7=fig.colorbar(cf, ax=axes[0,3])
        
        cf = PlotContour(axes[1,3], Entrainment_over_MF_eulerian_ratio_of_means, levels4, "<E>/<M>")
        cbar8=fig.colorbar(cf, ax=axes[1,3])
        
    # =========================
    # Formatting
    # =========================
    for axis in axes.ravel():
        apply_scientific_notation([axis])
        axis.set_ylim(ModelData.zh[0], ModelData.zh[-1])
    
    if plotType == "line":
        for axis in axes.ravel():
            axis.set_xlim(left=0)

    if plotType == "contour":
        cbars = [cbar1,cbar2,cbar3,cbar4]
        apply_scientific_notation_colorbar(cbars)
        cbars = [cbar5,cbar6,cbar7,cbar8]
        apply_scientific_notation_colorbar(cbars,decimals=3)
    
    return fig

In [ ]:
# step = 1 if ModelData.t_res == "5min" else 5
# # t1,t2 = 120*step, 122*step
# # t2,t2 = 90*step, 130*step
# t1,t2 = 0,ModelData.Ntime-1
# # t1,t2 = 45*step,110*step

# [Entrainment_mean, MF_lagrangian_mean,MF_eulerian_mean, Entrainment_over_MF_lagrangian_mean,Entrainment_over_MF_eulerian_mean,Entrainment_over_MF_lagrangian_ratio_of_means,Entrainment_over_MF_eulerian_ratio_of_means]\
# = CalculateMeans_MultipleTimes((t1,t2),updraftType="c")

# fig = PlotMeans(Entrainment_mean, MF_lagrangian_mean,MF_eulerian_mean, Entrainment_over_MF_lagrangian_mean,Entrainment_over_MF_eulerian_mean,Entrainment_over_MF_lagrangian_ratio_of_means,Entrainment_over_MF_eulerian_ratio_of_means)

# fig = PlotMeans(Entrainment_mean, MF_lagrangian_mean,MF_eulerian_mean, Entrainment_over_MF_lagrangian_mean,Entrainment_over_MF_eulerian_mean,Entrainment_over_MF_lagrangian_ratio_of_means,Entrainment_over_MF_eulerian_ratio_of_means,
#                plotType='contour')

In [ ]:
# step = 1 if ModelData.t_res == "5min" else 5
# # t1,t2 = 120*step, 122*step
# # t2,t2 = 90*step, 130*step
# # t1,t2 = 0,ModelData.Ntime-1
# t1,t2 = 45*step,110*step

# [Entrainment_mean, MF_lagrangian_mean,MF_eulerian_mean, Entrainment_over_MF_lagrangian_mean,Entrainment_over_MF_eulerian_mean,Entrainment_over_MF_lagrangian_ratio_of_means,Entrainment_over_MF_eulerian_ratio_of_means]\
# = CalculateMeans_MultipleTimes((t1,t2),updraftType="g")

# fig = PlotMeans(Entrainment_mean, MF_lagrangian_mean,MF_eulerian_mean, Entrainment_over_MF_lagrangian_mean,Entrainment_over_MF_eulerian_mean,Entrainment_over_MF_lagrangian_ratio_of_means,Entrainment_over_MF_eulerian_ratio_of_means)

# fig = PlotMeans(Entrainment_mean, MF_lagrangian_mean,MF_eulerian_mean, Entrainment_over_MF_lagrangian_mean,Entrainment_over_MF_eulerian_mean,Entrainment_over_MF_lagrangian_ratio_of_means,Entrainment_over_MF_eulerian_ratio_of_means,
#                plotType='contour')

In [ ]:
# def A_eulerian(t,updraftType="c"):
    
#     varName=f'A_{updraftType}'
#     A = CallVariable(ModelData, DataManager, ModelData.timeStrings[t], 
#                                                   variableName=varName)
#     return A*1

# parcelData=ModelData.OpenParcel()
# def GetSpatialData(t,wType="eulerian"):    
#     variableNames = ['Z', 'Y', 'X', 'W']
#     dataDictionary = MakeDataDictionary(variableNames,t)
#     Z,Y,X, W = (dataDictionary[k] for k in variableNames) #this W is eulerian w sampled onto lagrangian space
#     if wType=="eulerian":
#         return Z,Y,X, W
#     elif wType=="lagrangian":
#         W2 = parcelData['w'].isel(time=t).data #using lagrangian W instead of eulerian-lagrangian converted
#         return Z,Y,X, W2
        
# def A_lagrangian(t,updraftType="c"): #*MassFlux
#     """
#     lagrangian version
#     """
    
#     variableName=f'{Processed_string}A_{updraftType}'
#     A = CallLagrangianArray(ModelData, DataManager, ModelData.timeStrings[t], variableName=variableName, printstatement=False) 

#     [Z,Y,X,_]=GetSpatialData(t)

#     A_grid = np.zeros((ModelData.Nzh,ModelData.Nyh,ModelData.Nxh))
#     np.add.at(A_grid,(Z,Y,X),A)

#     A_grid[A_grid>=1]=1 #only interested in one count per grid
#     return A_grid


# #Calculating
# t=110
# t*=5 if ModelData.Nzh==95 else 1
# A_g_eulerian = A_eulerian(t,updraftType="g")
# A_c_eulerian = A_eulerian(t,updraftType="c")
# A_g_lagrangian = A_lagrangian(t,updraftType="g")
# A_c_lagrangian = A_lagrangian(t,updraftType="c")


# #Plotting
# a = np.sum(A_g_lagrangian,axis=(1,2))
# b = np.sum(A_c_lagrangian,axis=(1,2))
# c = np.sum(A_g_eulerian,axis=(1,2))
# d = np.sum(A_c_eulerian,axis=(1,2))

# fig,axes=plt.subplots(2, 2, figsize=(8,6))

# axis=axes[0,0]
# axis.plot(a,ModelData.zh,color='k')
# axis.set_xlim(left=0)
# axis.set_title("A_g (lagrangian)")

# axis=axes[0,1]
# axis.plot(b,ModelData.zh,color='k')
# axis.set_xlim(left=0)
# axis.set_title("A_c (lagrangian)")

# axis=axes[1,0]
# axis.plot(c,ModelData.zh,color='k')
# axis.set_xlim(left=0)
# axis.set_title("A_g (eulerian)")

# axis=axes[1,1]
# axis.plot(d,ModelData.zh,color='k')
# axis.set_xlim(left=0)
# axis.set_title("A_c (eulerian)")
# axis.set_xlabel('count')

# plt.tight_layout()

# for axis in axes.ravel():
#     axis.set_ylim(0,14)